<a href="https://colab.research.google.com/github/rohan-sarker-iem/Oracle/blob/main/Ellipse_Data_Points_CLUSTERING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""K-Means clustering demo for points sampled from ellipse-shaped groups.

This script uses only the Python standard library to generate synthetic 2D
points from several rotated ellipses and cluster them with a from-scratch
K-Means implementation. If Matplotlib is installed, it can also plot the result.
"""

from __future__ import annotations

import argparse
import math
import random
from dataclasses import dataclass
from typing import Iterable, Sequence

Point = tuple[float, float]


@dataclass(frozen=True)
class EllipseSpec:
    """Parameters describing one synthetic ellipse-shaped cluster."""

    center: Point
    radii: Point
    angle_degrees: float
    n_points: int
    noise: float = 0.08


def rotate_point(point: Point, angle_degrees: float) -> Point:
    """Rotate a 2D point by the supplied angle in degrees."""

    radians = math.radians(angle_degrees)
    cos_angle = math.cos(radians)
    sin_angle = math.sin(radians)
    x, y = point
    return x * cos_angle - y * sin_angle, x * sin_angle + y * cos_angle


def sample_ellipse_points(spec: EllipseSpec, rng: random.Random) -> list[Point]:
    """Generate points inside a rotated ellipse with light Gaussian noise."""

    points: list[Point] = []
    for _ in range(spec.n_points):
        # sqrt keeps the generated points uniformly distributed over ellipse area.
        theta = rng.uniform(0.0, 2.0 * math.pi)
        radius_scale = math.sqrt(rng.uniform(0.0, 1.0))
        unrotated = (
            spec.radii[0] * radius_scale * math.cos(theta),
            spec.radii[1] * radius_scale * math.sin(theta),
        )
        rotated_x, rotated_y = rotate_point(unrotated, spec.angle_degrees)
        points.append(
            (
                spec.center[0] + rotated_x + rng.gauss(0.0, spec.noise),
                spec.center[1] + rotated_y + rng.gauss(0.0, spec.noise),
            )
        )
    return points


def make_ellipse_dataset(
    specs: Iterable[EllipseSpec] | None = None,
    *,
    seed: int = 7,
) -> tuple[list[Point], list[int]]:
    """Create a synthetic dataset of ellipse-shaped clusters.

    Returns:
        A tuple of ``(points, true_labels)`` where each point is an ``(x, y)``
        pair and labels identify the ellipse that generated each point.
    """

    ellipse_specs = list(
        specs
        if specs is not None
        else (
            EllipseSpec(center=(-4.0, 1.5), radii=(2.8, 0.65), angle_degrees=25, n_points=180),
            EllipseSpec(center=(0.5, -2.0), radii=(0.8, 2.2), angle_degrees=-35, n_points=170),
            EllipseSpec(center=(4.0, 2.0), radii=(2.2, 0.9), angle_degrees=145, n_points=160),
        )
    )

    rng = random.Random(seed)
    points: list[Point] = []
    true_labels: list[int] = []
    for label, spec in enumerate(ellipse_specs):
        group = sample_ellipse_points(spec, rng)
        points.extend(group)
        true_labels.extend([label] * len(group))
    return points, true_labels


def squared_distance(first: Point, second: Point) -> float:
    """Return the squared Euclidean distance between two points."""

    return (first[0] - second[0]) ** 2 + (first[1] - second[1]) ** 2


def mean_point(points: Sequence[Point]) -> Point:
    """Return the coordinate-wise mean of a non-empty collection of points."""

    return sum(point[0] for point in points) / len(points), sum(point[1] for point in points) / len(points)


def initialize_centroids(points: Sequence[Point], k: int, rng: random.Random) -> list[Point]:
    """Choose ``k`` unique observations as the initial K-Means centroids."""

    if k <= 0:
        raise ValueError("k must be greater than zero")
    if k > len(points):
        raise ValueError("k cannot exceed the number of points")

    return rng.sample(list(points), k)


def assign_clusters(points: Sequence[Point], centroids: Sequence[Point]) -> list[int]:
    """Assign each point to the nearest centroid using Euclidean distance."""

    labels: list[int] = []
    for point in points:
        closest_index = min(range(len(centroids)), key=lambda index: squared_distance(point, centroids[index]))
        labels.append(closest_index)
    return labels


def update_centroids(
    points: Sequence[Point],
    labels: Sequence[int],
    centroids: Sequence[Point],
    rng: random.Random,
) -> list[Point]:
    """Move each centroid to the mean of its assigned points.

    Empty clusters are re-seeded to a random data point so the algorithm can
    continue without producing invalid centroids.
    """

    new_centroids: list[Point] = []
    for cluster_index in range(len(centroids)):
        cluster_points = [point for point, label in zip(points, labels) if label == cluster_index]
        if cluster_points:
            new_centroids.append(mean_point(cluster_points))
        else:
            new_centroids.append(rng.choice(points))
    return new_centroids


def centroid_shift(old_centroids: Sequence[Point], new_centroids: Sequence[Point]) -> float:
    """Return the total Euclidean movement between centroid updates."""

    return math.sqrt(
        sum(squared_distance(old_point, new_point) for old_point, new_point in zip(old_centroids, new_centroids))
    )


def k_means(
    points: Sequence[Point],
    k: int,
    *,
    max_iterations: int = 100,
    tolerance: float = 1e-4,
    seed: int = 7,
) -> tuple[list[int], list[Point], int]:
    """Cluster 2D points with K-Means.

    Args:
        points: Sequence of ``(x, y)`` observations.
        k: Number of clusters.
        max_iterations: Safety cap on update rounds.
        tolerance: Stop when centroid movement is below this amount.
        seed: Random seed used for centroid initialization and empty clusters.

    Returns:
        A tuple of ``(labels, centroids, iterations_run)``.
    """

    rng = random.Random(seed)
    centroids = initialize_centroids(points, k, rng)

    for iteration in range(1, max_iterations + 1):
        labels = assign_clusters(points, centroids)
        new_centroids = update_centroids(points, labels, centroids, rng)
        shift = centroid_shift(centroids, new_centroids)
        centroids = new_centroids
        if shift <= tolerance:
            break

    return assign_clusters(points, centroids), centroids, iteration


def plot_clusters(points: Sequence[Point], labels: Sequence[int], centroids: Sequence[Point]) -> None:
    """Display clustered points and final centroids with Matplotlib."""

    import matplotlib.pyplot as plt

    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    centroid_xs = [point[0] for point in centroids]
    centroid_ys = [point[1] for point in centroids]

    plt.figure(figsize=(8, 6))
    plt.scatter(xs, ys, c=labels, cmap="viridis", s=24, alpha=0.75)
    plt.scatter(
        centroid_xs,
        centroid_ys,
        c="red",
        marker="X",
        s=220,
        edgecolors="black",
        linewidths=1.2,
        label="Centroids",
    )
    plt.title("K-Means Clustering on Ellipse-Shaped Data")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.axis("equal")
    plt.legend()
    plt.tight_layout()
    plt.show()


def parse_args() -> argparse.Namespace:
    """Parse command-line arguments."""

    parser = argparse.ArgumentParser(description="Run K-Means on synthetic ellipse-shaped data.")
    parser.add_argument("--clusters", type=int, default=3, help="number of K-Means clusters")
    parser.add_argument("--seed", type=int, default=7, help="random seed for repeatable results")
    parser.add_argument("--max-iterations", type=int, default=100, help="maximum K-Means iterations")
    parser.add_argument("--no-plot", action="store_true", help="skip Matplotlib visualization")
    return parser.parse_args()


def main() -> None:
    """Generate ellipse data, run K-Means, print centroids, and optionally plot."""

    args = parse_args()
    points, true_labels = make_ellipse_dataset(seed=args.seed)
    predicted_labels, centroids, iterations = k_means(
        points,
        args.clusters,
        max_iterations=args.max_iterations,
        seed=args.seed,
    )

    print(f"Generated {len(points)} points from {len(set(true_labels))} ellipses.")
    print(f"K-Means converged in {iterations} iteration(s).")
    print("Final centroids:")
    for index, centroid in enumerate(centroids):
        print(f"  cluster {index}: x={centroid[0]:.3f}, y={centroid[1]:.3f}")

    if not args.no_plot:
        plot_clusters(points, predicted_labels, centroids)


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] [--clusters CLUSTERS] [--seed SEED]
                                [--max-iterations MAX_ITERATIONS] [--no-plot]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-7f17d94d-0eb5-4d13-8181-ba9881806ecb.json


SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import math
import random
import unittest

from oracle.elliptical_clustering import EllipticalGaussianMixture


def sample_bivariate_normal(rng, mean, covariance, size):
    # Cholesky factor for a 2x2 positive-definite covariance matrix.
    a = math.sqrt(covariance[0][0])
    b = covariance[0][1] / a
    c = math.sqrt(covariance[1][1] - b * b)
    points = []
    for _ in range(size):
        z1 = rng.gauss(0.0, 1.0)
        z2 = rng.gauss(0.0, 1.0)
        points.append((mean[0] + a * z1, mean[1] + b * z1 + c * z2))
    return points


class EllipticalGaussianMixtureTests(unittest.TestCase):
    def test_recovers_separated_elliptical_clusters(self):
        rng = random.Random(7)
        cluster_a = sample_bivariate_normal(rng, (-4.0, 0.0), ((4.0, 1.7), (1.7, 1.0)), 180)
        cluster_b = sample_bivariate_normal(rng, (4.0, 1.0), ((1.0, -0.8), (-0.8, 2.5)), 180)
        points = cluster_a + cluster_b
        truth = [0] * len(cluster_a) + [1] * len(cluster_b)

        model = EllipticalGaussianMixture(n_clusters=2, random_state=11, n_init=4)
        labels = model.fit_predict(points)

        accuracy = max(
            sum(label == expected for label, expected in zip(labels, truth)) / len(labels),
            sum((1 - label) == expected for label, expected in zip(labels, truth)) / len(labels),
        )
        self.assertGreater(accuracy, 0.95)
        self.assertEqual(len(model.means_), 2)
        self.assertEqual(len(model.covariances_), 2)
        for row in model.predict_proba(points):
            self.assertAlmostEqual(sum(row), 1.0)

    def test_rejects_invalid_inputs(self):
        with self.assertRaises(ValueError):
            EllipticalGaussianMixture(n_clusters=0)
        with self.assertRaises(ValueError):
            EllipticalGaussianMixture(n_clusters=2).fit([(1.0, 2.0), (float("nan"), 3.0)])
        with self.assertRaises(ValueError):
            EllipticalGaussianMixture(n_clusters=3).fit([(0.0, 0.0), (1.0, 1.0)])
        with self.assertRaises(ValueError):
            EllipticalGaussianMixture(n_clusters=1).fit([(1.0, 2.0, 3.0)])

    def test_prediction_requires_fit(self):
        model = EllipticalGaussianMixture(n_clusters=2)
        with self.assertRaises(RuntimeError):
            model.predict([(0.0, 0.0)])

    def test_information_criteria_are_finite(self):
        rng = random.Random(3)
        points = sample_bivariate_normal(rng, (0.0, 0.0), ((2.0, 0.6), (0.6, 1.0)), 80)
        estimator = EllipticalGaussianMixture(n_clusters=1, random_state=1)
        result = estimator.fit(points)
        self.assertTrue(math.isfinite(estimator.aic(points)))
        self.assertTrue(math.isfinite(estimator.bic(points)))
        self.assertEqual(len(result.labels), 80)


if __name__ == "__main__":
    unittest.main()


ModuleNotFoundError: No module named 'oracle'